# 01 - Ingesta de Facturas

**Objetivo**: Cargar archivos CSV de facturas mensuales y agregar metadata básica.

**Input**: `data/raw/{mes}/Facturas/*.csv`

**Output**: `data/staging/{mes}_raw.csv`

**Responsabilidades**:
- Listar archivos del mes
- Cargar cada CSV
- Extraer metadata del nombre de archivo (tienda, fecha)
- Agregar columnas de metadata
- Guardar todo en staging sin modificar los datos originales

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re

## Configuración

In [2]:
# Configurar el mes a procesar
MES = "enero"  # Cambiar según el mes
ANIO = 2026

# Rutas
RAW_DATA_PATH = Path("../data/raw") / MES / "Facturas"
STAGING_PATH = Path("../data/staging")
STAGING_PATH.mkdir(parents=True, exist_ok=True)

print(f"📁 Procesando facturas de: {MES} {ANIO}")
print(f"📂 Ruta raw: {RAW_DATA_PATH}")
print(f"💾 Ruta staging: {STAGING_PATH}")

📁 Procesando facturas de: enero 2026
📂 Ruta raw: ../data/raw/enero/Facturas
💾 Ruta staging: ../data/staging


## 1. Listar archivos del mes

In [3]:
# Listar todos los archivos CSV
csv_files = list(RAW_DATA_PATH.glob("*.csv"))

print(f"✅ Encontrados {len(csv_files)} archivos CSV:")
for file in sorted(csv_files):
    print(f"  - {file.name}")

✅ Encontrados 11 archivos CSV:
  - d1_03_enero.csv
  - d1_11_enero.csv
  - d1_15_enero.csv
  - d1_21_enero.csv
  - d1_27_enero.csv
  - mercacentro_06_enero.csv
  - mercacentro_21_enero.csv
  - surtiplaza_03_enero.csv
  - surtiplaza_09_enero.csv
  - surtiplaza_14_enero.csv
  - surtiplaza_19_enero.csv


## 2. Función para extraer metadata del nombre de archivo


In [4]:
# Mapeo de meses español -> número
MESES_MAP = {
    'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4,
    'mayo': 5, 'junio': 6, 'julio': 7, 'agosto': 8,
    'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12
}

def extraer_metadata_archivo(filename, mes_str, anio):
    """
    Extrae tienda y fecha del nombre de archivo.
    
    Formato esperado: {tienda}_{dia}_{mes}.csv
    Ejemplo: d1_11_enero.csv, surtiplaza_14_enero.csv
    """
    # Remover extensión .csv
    name = filename.stem
    
    # Dividir por guión bajo
    parts = name.split('_')
    
    if len(parts) < 3:
        raise ValueError(f"Formato de nombre incorrecto: {filename}")
    
    # Extraer tienda (todo antes del penúltimo elemento)
    tienda = '_'.join(parts[:-2])
    
    # Extraer día
    dia = int(parts[-2])
    
    # Crear fecha
    mes_num = MESES_MAP.get(mes_str.lower())
    if mes_num is None:
        raise ValueError(f"Mes no válido: {mes_str}")
    
    fecha = f"{anio}-{mes_num:02d}-{dia:02d}"
    
    return {
        'tienda': tienda,
        'fecha': fecha,
        'mes': mes_num,
        'año': anio
    }

# Probar la función
test_file = csv_files[0]
metadata = extraer_metadata_archivo(test_file, MES, ANIO)
print(f"\n🧪 Prueba con {test_file.name}:")
print(f"   Metadata extraída: {metadata}")


🧪 Prueba con d1_11_enero.csv:
   Metadata extraída: {'tienda': 'd1', 'fecha': '2026-01-11', 'mes': 1, 'año': 2026}


## 3. Cargar todos los archivos y agregar metadata

In [5]:
# Lista para almacenar todos los DataFrames
dfs_list = []

# Procesar cada archivo
for file in csv_files:
    print(f"\n📄 Procesando: {file.name}")
    
    # Cargar CSV
    df = pd.read_csv(file)
    print(f"   - Filas cargadas: {len(df)}")
    print(f"   - Columnas: {list(df.columns)}")
    
    # Extraer metadata
    metadata = extraer_metadata_archivo(file, MES, ANIO)
    
    # Agregar columnas de metadata
    for key, value in metadata.items():
        df[key] = value
    
    print(f"   - Metadata agregada: {metadata}")
    
    # Agregar a la lista
    dfs_list.append(df)

print(f"\n✅ Total de archivos procesados: {len(dfs_list)}")


📄 Procesando: d1_11_enero.csv
   - Filas cargadas: 8
   - Columnas: ['Producto', 'Cantidad', 'Unidad', 'Precio']
   - Metadata agregada: {'tienda': 'd1', 'fecha': '2026-01-11', 'mes': 1, 'año': 2026}

📄 Procesando: d1_03_enero.csv
   - Filas cargadas: 16
   - Columnas: ['Producto', 'Cantidad', 'Unidad', 'Precio']
   - Metadata agregada: {'tienda': 'd1', 'fecha': '2026-01-03', 'mes': 1, 'año': 2026}

📄 Procesando: surtiplaza_14_enero.csv
   - Filas cargadas: 11
   - Columnas: ['Item', 'Descripcion', 'Cantidad', 'Unidad', 'Valor_Unitario', 'Total']
   - Metadata agregada: {'tienda': 'surtiplaza', 'fecha': '2026-01-14', 'mes': 1, 'año': 2026}

📄 Procesando: surtiplaza_19_enero.csv
   - Filas cargadas: 5
   - Columnas: ['Item', 'Descripcion', 'Cantidad', 'Unidad', 'Valor_Unitario', 'Total']
   - Metadata agregada: {'tienda': 'surtiplaza', 'fecha': '2026-01-19', 'mes': 1, 'año': 2026}

📄 Procesando: mercacentro_06_enero.csv
   - Filas cargadas: 2
   - Columnas: ['Producto', 'Cantidad', 'Un

## 4. Combinar todos los DataFrames

In [6]:
# Combinar todos los DataFrames
df_raw = pd.concat(dfs_list, ignore_index=True)

print(f"📊 Dataset combinado:")
print(f"   - Total de filas: {len(df_raw)}")
print(f"   - Total de columnas: {len(df_raw.columns)}")
print(f"   - Columnas: {list(df_raw.columns)}")
print(f"\n🏪 Tiendas únicas: {df_raw['tienda'].unique()}")
print(f"📅 Fechas únicas: {sorted(df_raw['fecha'].unique())}")

📊 Dataset combinado:
   - Total de filas: 94
   - Total de columnas: 13
   - Columnas: ['Producto', 'Cantidad', 'Unidad', 'Precio', 'tienda', 'fecha', 'mes', 'año', 'Item', 'Descripcion', 'Valor_Unitario', 'Total', 'Ref']

🏪 Tiendas únicas: <StringArray>
['d1', 'surtiplaza', 'mercacentro']
Length: 3, dtype: str
📅 Fechas únicas: ['2026-01-03', '2026-01-06', '2026-01-09', '2026-01-11', '2026-01-14', '2026-01-15', '2026-01-19', '2026-01-21', '2026-01-27']


## 5. Vista previa del dataset

In [9]:
# Mostrar primeras filas
print("\n📋 Primeras 10 filas:")
df_raw.head(100)


📋 Primeras 10 filas:


,Producto,Cantidad,Unidad,Precio,tienda,fecha,mes,año,Item,Descripcion,Valor_Unitario,Total,Ref
0,Yogurt griego,1.00,und,7300.0,d1,2026-01-11,1,2026,NaN,NaN,NaN,NaN,NaN
1,Jamón Pietrán,1.00,und,13650.0,d1,2026-01-11,1,2026,NaN,NaN,NaN,NaN,NaN
2,Queso parmesano,1.00,und,14850.0,d1,2026-01-11,1,2026,NaN,NaN,NaN,NaN,NaN
3,Quesillo tajado,1.00,und,9990.0,d1,2026-01-11,1,2026,NaN,NaN,NaN,NaN,NaN
4,Topping mediano,4.00,und,19800.0,d1,2026-01-11,1,2026,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,Champiñón tajado 150g,1.00,und,5990.0,surtiplaza,2026-01-09,1,2026,NaN,NaN,NaN,NaN,NaN
90,Papaya común,2.57,kg,8687.0,surtiplaza,2026-01-09,1,2026,NaN,NaN,NaN,NaN,NaN
91,NaN,1.00,und,NaN,d1,2026-01-21,1,2026,1.0,YOGURT GRIEGO,7300.0,7300.0,NaN
92,NaN,3.00,und,NaN,d1,2026-01-21,1,2026,2.0,topping,14850.0,14850.0,NaN


In [10]:
# Información del DataFrame
print("\nℹ️ Información del dataset:")
df_raw.info()


ℹ️ Información del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Producto        56 non-null     str    
 1   Cantidad        94 non-null     float64
 2   Unidad          94 non-null     str    
 3   Precio          56 non-null     float64
 4   tienda          94 non-null     str    
 5   fecha           94 non-null     str    
 6   mes             94 non-null     int64  
 7   año             94 non-null     int64  
 8   Item            33 non-null     float64
 9   Descripcion     38 non-null     str    
 10  Valor_Unitario  38 non-null     float64
 11  Total           38 non-null     float64
 12  Ref             5 non-null      float64
dtypes: float64(6), int64(2), str(5)
memory usage: 9.7 KB


## 6. Guardar en staging

In [13]:
# Nombre del archivo de salida
output_file = STAGING_PATH / f"{MES}_{ANIO}_raw.csv"

# Guardar como CSV
df_raw.to_csv(output_file, index=False)

print(f"\n💾 Archivo guardado en: {output_file}")
print(f"   - Tamaño: {output_file.stat().st_size / 1024:.2f} KB")
print(f"\n✅ INGESTA COMPLETADA")


💾 Archivo guardado en: ../data/staging/enero_2026_raw.csv
   - Tamaño: 6.22 KB

✅ INGESTA COMPLETADA


## Resumen

Este notebook:
1. ✅ Cargó todos los archivos CSV del mes
2. ✅ Extrajo metadata (tienda, fecha) de los nombres de archivo
3. ✅ Agregó columnas de metadata a cada DataFrame
4. ✅ Combinó todos los DataFrames en uno solo
5. ✅ Guardó el resultado en `staging/` sin modificar los datos originales

**Siguiente paso**: Ejecutar `02_standardization.ipynb` para estandarizar el schema